In [1]:
import matplotlib.pyplot as plt
import numpy as np

from astropy.visualization import time_support
from astropy.time import Time
import astropy.units as u

from sunpy import timeseries as ts
from sunpy.net import Fido
from sunpy.net import attrs as a

from stixpy.net.client import STIXClient
from stixpy.timeseries import quicklook 
from stixpy.product import Product

import datetime as dt
from sunpy.time import parse_time
from sunpy.time import TimeRange

import pandas as pd

from scipy.signal import find_peaks, savgol_filter

from matplotlib import dates

In [5]:
def t(string):
    return parse_time(string).datetime

def get_energy_indices(qtable, ranges):
    energy_indices = []
    for e_min, e_max in ranges:
        start_index = np.where(qtable["e_low"] >= e_min)[0][0]
        end_index = np.where(qtable["e_high"] <= e_max)[0][-1]
        energy_indices.append([int(start_index), int(end_index)])
    return energy_indices
    
def get_stix_df(stix_sci, energy_ranges):
    
    energy_indices = get_energy_indices(stix_sci.energies, energy_ranges)
    counts, errors, times, timedeltas, energies = stix_sci.get_data(detector_indices=[[0, 31]],
                                                                    pixel_indices=[[0, 11]],
                                                                    energy_indices=energy_indices,)
    counts = counts.to(u.ct / u.s / u.keV)
    errors = errors.to(u.ct / u.s / u.keV)
    timedeltas = timedeltas.to(u.s)
    
    times = times
    
    energy_columns = [f"{e['e_low']}-{e['e_high']}" for e in energies]
    
    counts_reshaped = counts[:, 0, 0, :]
    
    counts_df = pd.DataFrame(counts_reshaped, index=times.datetime, columns=energy_columns)
    return counts_df

def rank_overlaps(start, end, sci_query):
    overlaps=[]
    for n in range(len(sci_query[0])):
        latest_start = max(start, sci_query[0][n][0].datetime)
        earliest_end = min(end, sci_query[0][n][1].datetime)
        
        overlap = earliest_end - latest_start
        overlaps.append((overlap.total_seconds(), n))
        
    return np.array(sorted(overlaps, key=lambda x: x[0], reverse=True))

In [4]:
energy_ranges = [(25.0*u.keV, 50*u.keV), (50.0*u.keV, 100*u.keV)]

In [34]:
flares = pd.read_csv("C:/Users/derva/OneDrive/Documents/DIAS/solar-orbiter-flare-statistics/data/STIX_flarelist_w_locations_20210101_20260130_version1_python.csv")
flarelist = flares.sort_values("25-50 keV", ascending=False).head(100)
flares.columns

Index(['start_UTC', 'end_UTC', 'peak_UTC', '4-10 keV', '10-15 keV',
       '15-25 keV', '25-50 keV', '50-84 keV', 'bkg 4-10 keV', 'bkg 10-15 keV',
       'bkg 15-25 keV', 'bkg 25-50 keV', 'bkg 50-84 keV',
       'bkg_baseline_4-10 keV', 'hpc_x_solo', 'hpc_y_solo', 'hpc_x_earth',
       'hpc_y_earth', 'visible_from_earth', 'hgs_lon', 'hgs_lat', 'hgc_lon',
       'hgc_lat', 'solo_position_lat', 'solo_position_lon',
       'solo_position_AU_distance', 'GOES_class_time_of_flare',
       'GOES_flux_time_of_flare', 'att_in', 'flare_id', 'sidelobes_ratio',
       'goes_estimated_min_class', 'goes_estimated_max_class',
       'goes_estimated_mean_class', 'goes_estimated_min_flux',
       'goes_estimated_max_flux', 'goes_estimated_mean_flux',
       'error_with_imaging', 'cpd_filename'],
      dtype='str')

In [35]:
flarelist

,start_UTC,end_UTC,peak_UTC,4-10 keV,10-15 keV,15-25 keV,25-50 keV,50-84 keV,bkg 4-10 keV,bkg 10-15 keV,...,flare_id,sidelobes_ratio,goes_estimated_min_class,goes_estimated_max_class,goes_estimated_mean_class,goes_estimated_min_flux,goes_estimated_max_flux,goes_estimated_mean_flux,error_with_imaging,cpd_filename
16121,2024-05-20T05:10:08.351,2024-05-20T06:52:32.361,2024-05-20 05:14:40.351,15995187,409599,884735,475135,204799,247.0,37.0,...,2405200514,0.750127,X6,X25,X12,5.834281e-04,0.002534,0.001198,False,./pixel_data/solo_L1_stix-sci-xray-cpd_2024052...
15798,2024-05-14T16:43:32.086,2024-05-14T17:02:16.088,2024-05-14 16:47:32.086,13148990,442367,753663,311295,155647,303.0,41.0,...,2405141647,0.754086,X4,X18,X8,4.217016e-04,0.001782,0.000854,False,./pixel_data/solo_L1_stix-sci-xray-cpd_2024051...
4360,2022-09-29T11:44:36.211,2022-09-29T12:51:48.218,2022-09-29 11:51:36.211,6411371,376831,507903,188415,77823,271.0,45.0,...,2209291151,0.798857,M7,X3,X1,7.000000e-05,0.000300,0.000100,False,./pixel_data/solo_L1_stix-sci-xray-cpd_2022092...
4361,2022-09-29T11:44:32.211,2022-09-29T12:51:56.218,2022-09-29 11:57:32.212,1769471,376831,507903,188415,77823,271.0,45.0,...,2209291157,0.800781,M2,M8,M4,2.000000e-05,0.000080,0.000040,False,./pixel_data/solo_L1_stix-sci-xray-cpd_2022092...
21278,2024-10-03T12:04:29.984,2024-10-03T13:15:41.992,2024-10-03 12:12:37.985,27216076,376831,622591,139263,34815,303.0,41.0,...,2410031212,0.878912,X1,X6,X3,1.490822e-04,0.000575,0.000289,False,./pixel_data/solo_L1_stix-sci-xray-cpd_2024100...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12385,2024-02-02T12:57:47.995,2024-02-02T13:58:08.001,2024-02-02 13:26:11.998,2148207,311295,69631,13823,3455,231.0,37.0,...,2402021326,0.757078,X1,X4,X2,1.057454e-04,0.000395,0.000201,False,./pixel_data/solo_L1_stix-sci-xray-cpd_2024020...
1349,2022-03-30T17:18:15.999,2022-03-30T18:46:12.008,2022-03-30 17:30:16.000,6478852,221183,59391,12799,1855,303.0,49.0,...,2203301730,0.793860,M5,X2,M9,5.000000e-05,0.000200,0.000090,False,./pixel_data/solo_L1_stix-sci-xray-cpd_2022033...
28196,2025-08-22T03:58:53.389,2025-08-22T04:04:53.390,2025-08-22 04:01:09.389,5509,3199,4863,12799,29695,303.0,41.0,...,2508220401,NaN,B8,C2,C1,8.214886e-07,0.000002,0.000001,True,./pixel_data/solo_L1_stix-sci-xray-cpd_2025082...
28194,2025-08-22T03:45:29.388,2025-08-22T03:49:13.388,2025-08-22 03:46:05.388,5509,2175,4351,12799,29695,303.0,41.0,...,2508220346,NaN,B8,C2,C1,8.216693e-07,0.000002,0.000001,True,./pixel_data/solo_L1_stix-sci-xray-cpd_2025082...


In [ ]:
df_list=[]
indices=[]
for ind in flarelist.index:
    sci_query = Fido.search(a.Time(flarelist['start_UTC'].loc[ind], 
                                   flarelist['end_UTC'].loc[ind]), 
                        a.Instrument.stix,
                        a.stix.DataType.sci,
                        a.stix.DataProduct.sci_xray_spec)
    sci_query['stix'].filter_for_latest_version()
    
    sci_files = Fido.fetch(sci_query)
    sci_files = sorted(sci_files)
    
    if len(sci_files)>=12
    sci_data_A = Product(sci_files[0])
    sci_data_B = Product(sci_files[1])
    
    df_A = get_stix_df(sci_data_A, energy_ranges)
    df_B = get_stix_df(sci_data_B, energy_ranges)
    
    combined_df = pd.concat([df_A, df_B])
    
    combined_df = combined_df.sort_index()
    
    df = combined_df[~combined_df.index.duplicated(keep="first")]

    df_list[ind]=df

    sci_df = get_stix_df(sci_data, energy_ranges)
    df_list.append(sci_df)
    indices.append(ind)

In [15]:
df_list=[]
indices=[]
testlengths=[]
for ind in top100.index:
    sci_query = Fido.search(a.Time(top100['start_UTC'].loc[ind], 
                                   top100['end_UTC'].loc[ind]), 
                        a.Instrument.stix,
                        a.stix.DataType.sci,
                        a.stix.DataProduct.sci_xray_spec)
    sci_query['stix'].filter_for_latest_version()
    testlengths.append(len(sci_query[0]))
    
    start = parse_time(top100['start_UTC'].loc[ind]).datetime
    end = parse_time(top100['end_UTC'].loc[ind]).datetime

    overlaps_ranked = rank_overlaps(start, end, sci_query)
    
    for index in overlaps_ranked[:,1]:
        badrange=False
        sci_files = Fido.fetch(sci_query[0][int(index)])
        sci_data = Product(sci_files)
    
        if sci_data.energies["e_high"][len(sci_data.energies["e_high"])-1]<100*u.keV or sci_data.energies["e_low"][0]>25*u.keV:
            badrange=True
        
        if not badrange:
            sci_df = get_stix_df(sci_data, energy_ranges)
            df_list.append(sci_df)
            indices.append(ind)
            break

Files Downloaded:   0%|          | 0/1 [00:00<?, ?file/s]

Files Downloaded:   0%|          | 0/1 [00:00<?, ?file/s]

Files Downloaded:   0%|          | 0/1 [00:00<?, ?file/s]

Files Downloaded:   0%|          | 0/1 [00:00<?, ?file/s]

Files Downloaded:   0%|          | 0/1 [00:00<?, ?file/s]

...

Files Downloaded:   0%|          | 0/1 [00:00<?, ?file/s]

Files Downloaded:   0%|          | 0/1 [00:00<?, ?file/s]

Files Downloaded:   0%|          | 0/1 [00:00<?, ?file/s]

Files Downloaded:   0%|          | 0/1 [00:00<?, ?file/s]

Files Downloaded:   0%|          | 0/1 [00:00<?, ?file/s]

In [26]:
list((np.where(np.array(testlengths)>1))[0])

[np.int64(9),
 np.int64(10),
 np.int64(26),
 np.int64(31),
 np.int64(39),
 np.int64(74),
 np.int64(84),
 np.int64(90),
 np.int64(95),
 np.int64(96)]

In [39]:
print(top100['start_UTC'].iloc[90], top100['end_UTC'].iloc[90])

2023-09-11T22:13:20.137 2023-09-11T23:36:00.146


In [38]:
sci_query = Fido.search(a.Time(top100['start_UTC'].iloc[90], 
                                   top100['end_UTC'].iloc[90]), 
                        a.Instrument.stix,
                        a.stix.DataType.sci,
                        a.stix.DataProduct.sci_xray_spec)
sci_query['stix'].filter_for_latest_version()
sci_query

Start Time,End Time,Instrument,Level,DataType,DataProduct,Ver,Request ID
Time,Time,str4,str2,str3,str13,str3,int64
2023-09-11T16:30:00.000,2023-09-11T22:35:00.000,STIX,L1,SCI,sci-xray-spec,V04,2309118146
2023-09-11T22:30:00.000,2023-09-12T04:35:01.000,STIX,L1,SCI,sci-xray-spec,V04,2309112494


In [12]:
i=0
for df in df_list:
    fig, ax = plt.subplots()
    ax.set_ylabel('ct/(keV s)')
    ax.set_title(f"{parse_time(flarelist['start_UTC'].iloc[i]).datetime.year}-{parse_time(flarelist['start_UTC'].iloc[i]).datetime.month}-{parse_time(flarelist['start_UTC'].iloc[i]).datetime.day} {parse_time(flarelist['start_UTC'].iloc[i]).datetime.hour}:{parse_time(flarelist['start_UTC'].iloc[i]).datetime.minute}")
    ax.xaxis.set_major_formatter(dates.DateFormatter("%H:%M"))
    
    ax.plot(df.index, df['25.0 keV-50.0 keV'])
    ax.plot(df.index, df['50.0 keV-100.0 keV'])
    
    handles, labels = ax.get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper right', fontsize='small')
    fig.savefig("C:/Users/derva/OneDrive/Documents/DIAS/solar-orbiter-flare-statistics/figures/week_6/top100_2550_counts/flare"+str(i)+".png", bbox_inches='tight')
    i+=1

plt.close('all')

C:\Users\derva\AppData\Local\Temp\ipykernel_26388\2096009873.py:3: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, ax = plt.subplots()
